In [29]:
import torch
import torch.nn as nn
import torchvision 

In [30]:
class Classifier(nn.Module):
    def __init__(self,backbone,feature_dim,num_classes):
        super().__init__()
        self.backbone=backbone


        self.head=nn.Linear(feature_dim,num_classes)
    def forward(self,x):
        feats=self.backbone(x)
        return self.head(feats)

In [31]:
def make_model(arch,num_classes,pretrained=True):
    if arch=="tinycnn":
        backbone = TinyCNNBackbone()
        feature_dim=128


    elif arch == "convenext_tiny":
        
        backbone=torchvision.models.convnext_tiny(pretrained=pretrained)
        feature_dim=backbone.classifier[2].in_features
        backbone.classifier=nn.Indentity()
        
    elif arch == "vit_b_16":
        
        backbone=torchvision.models.vit_b_16(pretrained=pretrained)
        feature_dim=backbone.heads.head.in_features
        backbone.heads=nn.Identity()
        
    elif arch== "faterrcnn_backbone":
        
        backbone=torchvision.models.detection.fasterrcnn_resnet50_fpn(
            pretrained=pretrained
        ).backbone
        feature_dim=256


    else:
        raise ValueError("Unknown architecture")


    return Classifier(backbone,feature_dim,num_classes)
        

In [32]:
import torch
import torch.nn as nn

class TinyCNNBackbone(nn.Module):
    def __init__(self):
        super().__init__()

        # TODO 1:
        # Define convolutional BLOCK 1
        # Input:  (B, 3, H, W)
        # Output: (B, 32, H/2, W/2)
        self.block1 = nn.Sequential(
            nn.Conv2d(3,32,kernel_size=3,padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2)
          
        )

        # TODO 2:
        # Define convolutional BLOCK 2
        # Output: (B, 64, H/4, W/4)
        self.block2 = nn.Sequential(
            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
         
        )

        # TODO 3:
        # Define convolutional BLOCK 3
        # Output: (B, 128, H/8, W/8)
        self.block3 = nn.Sequential(
            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        # TODO 4:
        # Global Average Pooling
        # This converts (B, 128, h, w) → (B, 128, 1, 1)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))

        # Optional but useful to expose
        self.out_features = 128

    def forward(self, x):
        # TODO 5:
        # Pass through block 1
        x = self.block1(x)

        # TODO 6:
        # Pass through block 2
        x = self.block2(x)

        # TODO 7:
        # Pass through block 3
        x = self.block3(x)

        # TODO 8:
        # Apply global average pooling
        x = self.gap(x)

        # TODO 9:
        # Flatten to (B, 128)
        x = torch.flatten(x, 1)

        return x


In [33]:
model = TinyCNNBackbone()
x = torch.randn(4, 3, 32, 32)
y = model(x)
print(y.shape)


torch.Size([4, 128])


In [34]:
model=make_model(arch="tinycnn",num_classes=10,pretrained=True)

In [36]:
x = torch.randn(2, 3, 32, 32)
y = model(x)
print(y.shape)


torch.Size([2, 10])
